In [1]:
import os
from pathlib import Path
import numpy as np
import tifffile as tiff, zarr

In [4]:
IMAGE_PATH = "/data/work/DATA_UJI/HE_OpenSource/B04372C214_HE_regist.tif"      
OUT_DIR    = "/data/work/DATA_UJI/B04372C214_Tiles" 
PATCH      = 512
STRIDE     = 384                             
SAVE_DTYPE = np.uint8
REPLICATE_GRAY_TO_RGB = True  

In [5]:
with tiff.TiffFile(IMAGE_PATH) as tf:
    s = tf.series[0]
    axes  = s.axes        # mis. 'YXS' (H,W,Channels) atau 'CYX' (Channels,H,W) dsb.
    store = s.aszarr()    # zarr store untuk akses window per-patch (hemat RAM)
z = zarr.open(store, mode="r")
shape = z.shape
print("Axes:", axes, "| shape:", shape)

Axes: YXS | shape: (11760, 11760, 3)


In [6]:
def idx(ax):
    return axes.index(ax) if ax in axes else None

iy = idx('Y'); ix = idx('X')
ic = idx('C'); isamp = idx('S')     # sebagian TIFF pakai 'S' utk SamplesPerPixel
ichannel = ic if ic is not None else isamp

H = shape[iy]; W = shape[ix]

In [7]:
def starts_with_edges(length, patch, stride):
    starts = list(range(0, max(1, length - patch + 1), stride))
    last = length - patch
    if starts[-1] != last:
        starts.append(last)  # paksa anchor terakhir = tepi
    return starts

In [8]:
xs = starts_with_edges(W, PATCH, STRIDE)
ys = starts_with_edges(H, PATCH, STRIDE)
print(f"Tiles: {len(xs)} x {len(ys)} = {len(xs)*len(ys)}")
print(f"X first={xs[0]}, last={xs[-1]} (harus {W-PATCH}); Y first={ys[0]}, last={ys[-1]} (harus {H-PATCH})")

Tiles: 31 x 31 = 961
X first=0, last=11248 (harus 11248); Y first=0, last=11248 (harus 11248)


In [9]:
def read_window_rgb(y0,y1,x0,x1):
    # siapkan slice semua dimensi
    sl = [slice(None)] * z.ndim
    sl[iy] = slice(y0,y1)
    sl[ix] = slice(x0,x1)
    tile = z[tuple(sl)]

    # Kalau ada sumbu kanal, pindahkan ke belakang (HWC)
    if ichannel is not None:
        pos_c = axes.index('C') if 'C' in axes else axes.index('S')
        if pos_c != tile.ndim - 1:
            tile = np.moveaxis(tile, pos_c, -1)
        # jika lebih dari 3 kanal, ambil 3 kanal pertama
        if tile.shape[-1] > 3:
            tile = tile[...,:3]

    # Jika tidak ada kanal (grayscale, 2D), optional: replikasikan ke 3 kanal agar "RGB"
    if tile.ndim == 2 and REPLICATE_GRAY_TO_RGB:
        tile = np.stack([tile, tile, tile], axis=-1)  # (H,W) -> (H,W,3)

    return tile

In [10]:
# ====== TILING & SIMPAN ======
out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
count = 0
for yi, y0 in enumerate(ys):
    for xi, x0 in enumerate(xs):
        y1, x1 = y0 + PATCH, x0 + PATCH
        tile = read_window_rgb(y0,y1,x0,x1)

        # pastikan dtype konsisten
        if tile.dtype != SAVE_DTYPE:
            if tile.dtype == np.uint16 and SAVE_DTYPE == np.uint8:
                tile = (tile / 256).astype(np.uint8)  # scale 16-bit -> 8-bit
            else:
                tile = tile.astype(SAVE_DTYPE)

        fname = f"patch_{count:06d}_r{yi:03d}_c{xi:03d}_y{y0:05d}_x{x0:05d}.tif"
        tiff.imwrite(
            str(out_dir / fname),
            tile,
            compression="zlib",  # lossless
            bigtiff=True         # aman untuk banyak file/ukuran besar (BigTIFF)
        )
        count += 1

In [11]:
print("Selesai. Tiles ditulis:", count)

Selesai. Tiles ditulis: 961
